# Phân tích yêu cầu chi phí

Sổ tay này trình bày cách tạo các tác nhân sử dụng plugin để xử lý chi phí đi lại từ hình ảnh biên lai địa phương, tạo email yêu cầu chi phí, và trực quan hóa dữ liệu chi phí bằng biểu đồ hình tròn. Các tác nhân chọn chức năng một cách động dựa trên ngữ cảnh nhiệm vụ.

Các bước:
1. Tác nhân OCR xử lý hình ảnh biên lai địa phương và trích xuất dữ liệu chi phí đi lại.
2. Tác nhân Email tạo ra email yêu cầu chi phí.

### Ví dụ về kịch bản chi phí đi lại:
Hãy tưởng tượng bạn là nhân viên đi công tác đến một thành phố khác. Công ty bạn có chính sách hoàn trả tất cả các chi phí liên quan đến đi lại hợp lý. Dưới đây là phân tích các chi phí đi lại tiềm năng:
- Vận chuyển:
Vé máy bay khứ hồi từ thành phố nhà bạn đến thành phố đích.
Taxi hoặc dịch vụ gọi xe đến và đi từ sân bay.
Phương tiện giao thông địa phương tại thành phố đích (như phương tiện giao thông công cộng, xe thuê, hoặc taxi).

- Lưu trú:
Ở khách sạn ba đêm tại một khách sạn kinh doanh tầm trung gần địa điểm họp.

- Ăn uống:
Trợ cấp ăn uống hàng ngày cho bữa sáng, trưa và tối, dựa trên chính sách trợ cấp theo ngày của công ty.

- Chi phí khác:
Phí đậu xe tại sân bay.
Phí truy cập Internet tại khách sạn.
Tiền bo hoặc các khoản phí dịch vụ nhỏ.

- Hồ sơ:
Bạn nộp tất cả biên lai (vé máy bay, taxi, khách sạn, ăn uống, v.v.) và một báo cáo chi phí đã hoàn chỉnh để được hoàn trả.


## Import required libraries

Nhập các thư viện và mô-đun cần thiết cho sổ tay.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Định nghĩa Mô hình Chi tiêu

 Tạo một mô hình Pydantic cho các khoản chi tiêu cá nhân và một lớp ExpenseFormatter để chuyển đổi truy vấn của người dùng thành dữ liệu chi tiêu có cấu trúc.

 Mỗi khoản chi tiêu sẽ được biểu diễn theo định dạng:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Định nghĩa Công cụ - Tạo Email

Tạo một hàm công cụ để tạo email gửi yêu cầu hoàn trả chi phí.
- Công cụ này sử dụng decorator `@tool` từ Microsoft Agent Framework.
- Nó tính tổng số tiền chi phí và định dạng chi tiết thành nội dung email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Công cụ Trích xuất Chi phí Du lịch từ Hình ảnh Biên lai

Tạo một hàm công cụ để trích xuất chi phí du lịch từ hình ảnh biên lai.
- Công cụ này sử dụng bộ trang trí `@tool` từ Microsoft Agent Framework.
- Nó đọc hình ảnh biên lai, mã hóa nó thành base64, và trả về URI dữ liệu để đại lý phân tích.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Xử lý Chi phí

Định nghĩa các tác nhân và kết nối chúng thành một quy trình tuần tự sử dụng `WorkflowBuilder`.
- Tác nhân OCR trích xuất dữ liệu chi phí có cấu trúc từ ảnh hóa đơn sử dụng công cụ `load_receipt_image`.
- Tác nhân Email lấy dữ liệu đã trích xuất và tạo email yêu cầu hoàn chi phí chuyên nghiệp sử dụng công cụ `generate_expense_email`.
- `WorkflowBuilder` với `add_edge` tạo đường ống tuần tự: Tác nhân OCR → Tác nhân Email.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hàm chính

Xây dựng quy trình tuần tự và chạy nó để xử lý hình ảnh biên lai và tạo email yêu cầu chi phí.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi luôn cố gắng đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc không chính xác. Tài liệu gốc bằng ngôn ngữ gốc của nó vẫn được coi là nguồn tin chính xác và đáng tin cậy. Đối với thông tin quan trọng, chúng tôi khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp của con người. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu nhầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
